### [Quick Start Guide](https://ankandrew.github.io/fast-alpr/latest/quick_start/) for ALPR Package ###

**Import Statements**

Functions Imported from ocr_utils:

* crop_img_yolo

* aplr_single_image

* annotate_images_ocr

* review_image

In [ ]:
import os
import sys
import os
import numpy as np
from fast_alpr import ALPR
import pandas as pd
import cv2
import seaborn as sns

# Get the path of the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add it to the Python search path if it isn't there already
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from utils.ocr_utils import alpr_single_image, do_alpr, review_image

**Define Image and Annotation paths**

In [ ]:
# folder path to training data
train_img_path = r"../data/license_plate_detection/train/images"
train_anno_path = r"../data/license_plate_detection/train/labels"

**Run ALPR model on single image**

In [ ]:
alpr = ALPR()

test_image= r"../data/license_plate_detection/train/images/lp_train_972.jpg"
test_cv2 = cv2.imread(test_image)

alpr_single_image(alpr, test_cv2,"lp_train_972")

**Main OCR Function**

* runs set of images and annotations through OCR process
* returns dataframe with OCR predictions for each image

In [ ]:
output_df = do_alpr(train_anno_path, train_img_path)

**Save result to CSV**

In [ ]:
save_path = r"..\data\alpr_ocr_results.csv"

output_df.to_csv(save_path)

**Summary Statistics**

In [ ]:
import ast

ocr_df = pd.read_csv(r"..\data\alpr_ocr_results.csv")

def func(input_str):
    if pd.isna(input_str):
        return float('nan')
    
    inp_list = ast.literal_eval(input_str)
    fl_list = [item for item in inp_list if isinstance(item, float)]

    return sum(fl_list)/len(fl_list)

def func1(input_str):
    if pd.isna(input_str):
        return float('nan')
    
    
    inp_list = ast.literal_eval(input_str)
    if len(inp_list) == 4:

        ctr_x = (inp_list[0]+inp_list[1])/2
        ctr_y = (inp_list[2]+inp_list[3])/2

    return (ctr_x, ctr_y)

def func2(input_str):
    if pd.isna(input_str):
        return float('nan')
    
    
    inp_list = ast.literal_eval(input_str)
    if len(inp_list) == 4:

        return(inp_list[1]-inp_list[0])*(inp_list[3]-inp_list[2])

total = ocr_df.shape[0]
na_counts = ocr_df['text'].isna().sum()
na_region_counts = ocr_df['region'].isna().sum()

print(f"Regions Detected: {(1-na_region_counts/total)*100:.2f}% \n")
print(f"Licence plates with extracted text: {(1-na_counts/total)*100:.2f}% \n")

print(f"Regions detected and value counts: {ocr_df["region"].value_counts()}")

ocr_df = ocr_df.dropna(subset= ["text"])
ocr_df["avg_confidence"] = ocr_df["confidence"].apply(func)
ocr_df["center_norm"] = ocr_df["bbox_norm"].apply(func1)
ocr_df["area_norm"] = ocr_df["bbox_norm"].apply(func2)

**Histplot of area of OCR within bounding box**

In [ ]:
sns.histplot(ocr_df, x = "area_norm")

**Center coords in bounding box**

In [ ]:
ocr_df[['x_norm', 'y_norm']] = pd.DataFrame(ocr_df['center_norm'].to_list(), index=ocr_df.index)
sns.scatterplot(ocr_df, x = 'x_norm', y = 'y_norm', hue = "area_norm")

**Analysis of Reviewed Images after running through OCR**

In [ ]:
reviewed_df = pd.read_csv(r"..\data\ocr_review_results.csv")

val_counts = reviewed_df["Review_Status"].value_counts(normalize=True).to_dict()

accuracy = val_counts["Yes"]
print(f"Accuracy over {reviewed_df.shape[0]} Images: {(accuracy*100):.2f}%")
#print(val_counts["count"])

**Filter results by lowest confidence score < a threshold value and evaluate accuracy**

In [ ]:
# merge review_results df and ocr results df
reviewed_df["Original_Image"] = reviewed_df["Original_Image"].apply(lambda x: x.replace(".jpg",""))
format_df = reviewed_df.rename(columns={"Original_Image":"file_name"})
combo_df = pd.merge(left = ocr_df, right = format_df, how = "outer", on = "file_name")

# filter to only rows that have a Y/N in the review status column
combo_df = combo_df.dropna(subset = "Review_Status")

# threshold value
thresh = .90

def conf_filt(inp_list):
    inp_list = ast.literal_eval(inp_list)
    return(min(inp_list))

# create new column with value of lowest confidence 
combo_df["lowest_conf"] = combo_df["confidence"].apply(conf_filt)

# filter to lowest confidence values above threshold
thresh_df = combo_df[combo_df["lowest_conf"] > thresh]

# value counts
val_counts_thresh = thresh_df["Review_Status"].value_counts(normalize=True).to_dict()

# accuracy
accuracy_thresh = val_counts_thresh["Yes"]
print(f"Accuracy over {thresh_df.shape[0]} Images: {(accuracy_thresh*100):.2f}%")